## 1) Install dependencies

In [1]:
!pip -q install --upgrade pip
!pip -q install openai requests scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.9 MB/s eta 0:00:0000:0100:01


## 2) Clone repository (if needed) and set working directory

In [2]:
import os
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
CLONE_DIR = WORK_DIR / "tonmoy99_create-RL-Brain"

if not CLONE_DIR.exists():
    !git clone https://github.com/Tonmoy221/tonmoy99_create-RL-Brain.git "{CLONE_DIR.name}"

search_roots = [
    WORK_DIR / "tonmoy99_Vedio-Gen",
    CLONE_DIR,
]

matches = []
for root in search_roots:
    if root.exists():
        matches.extend(root.rglob("llm_pipeline.py"))

if not matches:
    raise FileNotFoundError(
        "llm_pipeline.py not found under /kaggle/working. "
        "Expected in your project folder after clone."
    )

# Prefer the intended project folder if multiple matches exist.
matches = sorted(
    matches,
    key=lambda p: (
        "tonmoy99_Vedio-Gen" not in str(p.parent),
        len(str(p.parent)),
    ),
)

script_path = matches[0]
repo_path = script_path.parent

os.chdir(repo_path)
print("Current working directory:", os.getcwd())
print("Using llm_pipeline.py:", script_path)
print("llm_pipeline.py exists:", Path("llm_pipeline.py").exists())

Cloning into 'tonmoy99_create-RL-Brain'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 81 (delta 21), reused 77 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 467.33 KiB | 5.99 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Current working directory: /kaggle/working/tonmoy99_create-RL-Brain
Using llm_pipeline.py: /kaggle/working/tonmoy99_create-RL-Brain/llm_pipeline.py
llm_pipeline.py exists: True


## 3) Configure API credentials (safe way)
Use Kaggle Secrets for `OPENROUTER_API_KEY` instead of hardcoding keys in notebook cells.

In [3]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise RuntimeError("Missing Kaggle secret OPENROUTER_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["OPENAI_HTTP_REFERER"] = "https://www.kaggle.com"
os.environ["OPENAI_X_TITLE"] = "LLM Agent Pipeline Test"

print("Environment configured for OpenRouter through OpenAI-compatible client.")

Environment configured for OpenRouter through OpenAI-compatible client.


## 4) Direct LLM smoke test (OpenRouter)
This verifies API/model is reachable and generates a cinematic seed prompt.

In [4]:
from openai import OpenAI

smoke_client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

smoke_completion = smoke_client.chat.completions.create(
    extra_headers={
        "HTTP-Referer": os.environ.get("OPENAI_HTTP_REFERER", "https://www.kaggle.com"),
        "X-Title": os.environ.get("OPENAI_X_TITLE", "LLM Agent Pipeline Test"),
    },
    model="google/gemma-3n-e4b-it:free",
    messages=[
        {
            "role": "user",
            "content": "Generate one cinematic seed prompt for a 4-scene short film with strong character and location continuity.",
        }
    ],
)

seed_prompt = smoke_completion.choices[0].message.content.strip()
print("Generated seed prompt:\n")
print(seed_prompt)

Generated seed prompt:

## Cinematic Seed Prompt: The Clockwork Heart

**Logline:** In a secluded, rain-soaked clockwork city powered by intricate automatons, a solitary clocksmith must confront his past and the city's fading magic when a young, curious apprentice discovers a hidden truth within a broken automaton heart.

**Visual Style:**  Dieselpunk meets Gothic fantasy.  Think *Brazil* meets *Pan's Labyrinth*.  Heavy use of brass, gears, steam, and flickering gaslight.  Color palette: Cool blues, deep greens, and rust-colored browns, punctuated by the warm glow of mechanical lights.

**Core Themes:**  Memory, loss, legacy, the dangers of clinging to the past, finding hope in unexpected places.


**Scene Breakdown & Character/Location Continuity:**

*   **Scene 1: The Workshop (Establishing Shot)** - Wide, establishing shot of the clockwork city. Rain is pouring. Focus on the intricate architecture and steam rising from various mechanisms.  We then zoom in to a cluttered workshop, il

## 5) Run the LLM Agent Pipeline (no WAN)
Runs your standalone `llm_pipeline.py` and writes artifacts.

In [ ]:
import sys, os, traceback, time, random

# Seed prompt – the pipeline will expand this into full character appearances + scene breakdown
seed_prompt = (
    "A lone astronaut drifts through the crystalline rings of Saturn, "
    "helmet visor reflecting a dying sun."
)

# Locate the cloned repo
repo_root = None
for candidate in [
    "/kaggle/working/tonmoy99_create-RL-Brain",
    "/kaggle/working",
]:
    p = Path(candidate)
    if (p / "llm_pipeline.py").exists():
        repo_root = p
        break
if repo_root is None:
    for hit in Path("/kaggle/working").rglob("llm_pipeline.py"):
        repo_root = hit.parent
        break
if repo_root is None:
    raise FileNotFoundError("Cannot find llm_pipeline.py in /kaggle/working")

print("Using repo at:", repo_root)
os.chdir(str(repo_root))

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    import llm_pipeline as lp
    from openai import OpenAI, RateLimitError as _OAIRateLimit

    # ── Retry settings for free-tier rate limits ──
    MAX_RETRIES = 8
    BASE_DELAY  = 15   # seconds
    MAX_DELAY   = 120  # seconds

    def _openai_compat_call(self, system_prompt: str, user_prompt: str) -> str:
        """Drop-in replacement for LLMDirectorAgent._call_openai with:
        1) Exponential-backoff retry on 429 rate-limit errors
        2) System-message fallback for models that reject them
        """
        api_key = os.getenv("OPENAI_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError("Missing OPENAI_API_KEY environment variable")

        base_url = os.getenv("OPENAI_BASE_URL", "").strip() or None
        client_kwargs = {"api_key": api_key}
        if base_url is not None:
            client_kwargs["base_url"] = base_url

        client = OpenAI(**client_kwargs)

        extra_headers = {}
        http_referer = os.getenv("OPENAI_HTTP_REFERER", "").strip()
        x_title = os.getenv("OPENAI_X_TITLE", "").strip()
        if http_referer:
            extra_headers["HTTP-Referer"] = http_referer
        if x_title:
            extra_headers["X-Title"] = x_title

        # Build two possible message lists
        msgs_system = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        merged_user_prompt = (
            "Follow these instructions exactly:\n"
            f"{system_prompt}\n\n"
            "User request:\n"
            f"{user_prompt}"
        )
        msgs_user_only = [{"role": "user", "content": merged_user_prompt}]

        use_system = True  # start by trying system+user

        for attempt in range(1, MAX_RETRIES + 1):
            messages = msgs_system if use_system else msgs_user_only
            try:
                response = client.chat.completions.create(
                    model=self.model_name,
                    temperature=0.4,
                    messages=messages,
                    extra_headers=extra_headers if extra_headers else None,
                )
                return response.choices[0].message.content or ""

            except _OAIRateLimit as rate_err:
                # 429 – wait with exponential backoff + jitter then retry
                delay = min(BASE_DELAY * (2 ** (attempt - 1)), MAX_DELAY) + random.uniform(0, 5)
                print(f"  [RATE-LIMIT] attempt {attempt}/{MAX_RETRIES}, "
                      f"sleeping {delay:.1f}s …")
                time.sleep(delay)

            except Exception as exc:
                text = str(exc).lower()
                if "developer instruction is not enabled" in text and use_system:
                    # Switch to user-only messages and retry immediately
                    print("  [FALLBACK] system role rejected → using user-only messages")
                    use_system = False
                    continue
                raise

        raise RuntimeError(
            f"OpenRouter rate-limited after {MAX_RETRIES} retries. "
            "Wait a few minutes and re-run this cell."
        )

    # Runtime monkey patch for compatibility in Kaggle clone.
    lp.LLMDirectorAgent._call_openai = _openai_compat_call

    report_out = lp.run_llm_pipeline(
        seed_prompt=seed_prompt,
        output_root=".",
        provider="openai",
        model_name="google/gemma-3n-e4b-it:free",
        resume=False,
    )
    print("Pipeline completed successfully.")
    print("Report path:", report_out)
except Exception as exc:
    print("Pipeline failed with exception:", str(exc))
    print("Detailed traceback:\n")
    traceback.print_exc()
    raise


Using repo at: /kaggle/working/tonmoy99_create-RL-Brain
LLM-ONLY CINEMATIC AGENT PIPELINE
[INFO] Starting fresh LLM-only run
[STEP] Generate creative document
  [FALLBACK] system role rejected → using user-only messages
  [FALLBACK] system role rejected → using user-only messages
[STEP] Scene prompt refinement loop
  [RATE-LIMIT] attempt 1/8, sleeping 18.9s …
  [FALLBACK] system role rejected → using user-only messages
  [FALLBACK] system role rejected → using user-only messages
[SCENE scene_1] score=0.9000 accept=True
  [FALLBACK] system role rejected → using user-only messages
  [RATE-LIMIT] attempt 1/8, sleeping 17.8s …
  [RATE-LIMIT] attempt 2/8, sleeping 32.9s …
  [FALLBACK] system role rejected → using user-only messages
[SCENE scene_2] score=0.8500 accept=True
  [FALLBACK] system role rejected → using user-only messages
  [FALLBACK] system role rejected → using user-only messages
[SCENE scene_3] score=0.9500 accept=True
  [FALLBACK] system role rejected → using user-only message

## 6) Validate pipeline artifacts and pass/fail status

In [6]:
import json
from pathlib import Path

creative_doc_path = Path("story_bible/llm_only/creative_document_llm.json")
scene_prompts_path = Path("output/llm_only/scene_prompts_llm.json")
report_path = Path("output/llm_only/llm_pipeline_report.json")
memory_path = Path("memory_llm/state_llm.json")

required_paths = [creative_doc_path, scene_prompts_path, report_path, memory_path]
missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    raise FileNotFoundError(f"Missing expected artifacts: {missing}")

creative_doc = json.loads(creative_doc_path.read_text(encoding="utf-8"))
scene_prompts = json.loads(scene_prompts_path.read_text(encoding="utf-8"))
report = json.loads(report_path.read_text(encoding="utf-8"))
memory_state = json.loads(memory_path.read_text(encoding="utf-8"))

scene_count = len(creative_doc.get("scenes", []))
mean_score = float(report.get("mean_critique_score", 0.0))
continuity_count = len(memory_state.get("continuity_log", []))

print("PASS: LLM pipeline artifacts generated")
print("scene_count:", scene_count)
print("mean_critique_score:", round(mean_score, 4))
print("continuity_log_count:", continuity_count)
print("report:", report_path)

PASS: LLM pipeline artifacts generated
scene_count: 4
mean_critique_score: 0.9125
continuity_log_count: 4
report: output/llm_only/llm_pipeline_report.json


## 7) Embedded result export (vector form)
Convert generated prompts into numeric vectors and save as artifact before WAN.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [item.get("prompt", "") for item in scene_prompts if item.get("prompt")]
if not texts:
    raise ValueError("No prompts found in scene_prompts_llm.json")

vectorizer = TfidfVectorizer(max_features=64)
embeddings = vectorizer.fit_transform(texts).toarray()

embedded_rows = []
for idx, row in enumerate(scene_prompts, start=1):
    vec = embeddings[idx - 1].tolist() if idx - 1 < len(embeddings) else []
    embedded_rows.append(
        {
            "scene_id": row.get("scene_id"),
            "prompt": row.get("prompt", ""),
            "critique_score": row.get("critique", {}).get("score", 0.0),
            "scene_character_assignments": row.get("scene_character_assignments", {}),
            "embedding": vec,
        }
    )

embedded_path = Path("output/llm_only/scene_prompts_embedded.json")
embedded_path.write_text(json.dumps(embedded_rows, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Scene summary preview ──
preview_rows = []
for r in embedded_rows:
    aliases = r.get("scene_character_assignments", {})
    alias_str = ", ".join(f"{v}" for v in aliases.values()) if aliases else "—"
    preview_rows.append({
        "scene_id": r["scene_id"],
        "critique_score": r["critique_score"],
        "embedding_dim": len(r["embedding"]),
        "character_aliases": alias_str,
    })

preview = pd.DataFrame(preview_rows)

# ── Character physical appearance summary ──
char_rows = []
for char in creative_doc.get("characters", []):
    char_rows.append({
        "id": char.get("id", ""),
        "name": char.get("name", ""),
        "gender": char.get("gender", ""),
        "age": char.get("age", ""),
        "skin_tone": char.get("skin_tone", ""),
        "body_type": char.get("body_type", ""),
        "height": char.get("height", ""),
        "hair": f"{char.get('hair_color', '')} / {char.get('hair_style', '')}",
        "eye_color": char.get("eye_color", ""),
        "expression": char.get("facial_expression", ""),
        "features": char.get("distinguishing_features", ""),
        "wardrobe": char.get("wardrobe", ""),
    })
char_preview = pd.DataFrame(char_rows)

print("Embedded artifact saved:", embedded_path)
print("\n── Scene prompt overview ──")
display(preview)
print("\n── Character physical appearance registry ──")
display(char_preview)


Embedded artifact saved: output/llm_only/scene_prompts_embedded.json


,scene_id,critique_score,embedding_dim
0,scene_1,0.90,64
1,scene_2,0.85,64
2,scene_3,0.95,64
3,scene_4,0.95,64
